In [ ]:
import openai
import os
import pandas as pd
import json
data_path = "./misSpans_zs.json"

with open(data_path, encoding="utf-8") as file:
    data_json = json.loads(file.read())
len(data_json)

In [ ]:
output_path = "./evaluation.csv"
from openai import OpenAI
api_key = ""
base_url = "https://api.deepseek.com"

client = OpenAI(
    base_url=base_url,
    api_key=api_key,
)


def get_response(query):
    completion = client.chat.completions.create(
      model="deepseek-reasoner",
      seed = 42,
      messages=query,
      temperature=0,
      stream=False,
      max_completion_tokens=8192,
    )

    reasoning_content = completion.choices[0].message.reasoning_content
    content = completion.choices[0].message.content
    return reasoning_content, content


init_data = {}


for index, row in enumerate(data_json):
    print(index)
    if index < 0:
        continue
    conversation_list = []
    instruction_prefix = """Based on the following information of a case to answer. When responding, ensure that the final section contains the complete answer and uses the format: Answer: [Your answer] \n"""

    instruction_text = instruction_prefix + row["instruction"]
    conversation_list.append({"role":"user","content":instruction_text})
    row["content"] = instruction_text
    
    print(conversation_list)
    try:
        row["reasoning"], row["generated_answer"] = get_response(conversation_list)
    except Exception as e:
        print("Errertype:", e)
        print("TimeoutError sleep and reget the response....")
        time.sleep(10)
        row["reasoning"], row["generated_answer"] = get_response(conversation_list)
    print("chatgpt:", row["reasoning"], row["generated_answer"])


    if index == 0:
        pd.DataFrame([row]).to_csv(output_path, mode="a", index=False)
    else:
        pd.DataFrame([row]).to_csv(output_path, mode="a", header=False, index=False)